# ProstT5 Inverse Folding — Baseline Performance (3Di → AA)

This notebook establishes the **baseline performance** for inverse-folding (3Di → AA) for two pipelines:

1. **enc–dec**: the full `Rostlab/ProstT5_fp16` encoder–decoder, autoregressive generation with greedy decoding.
2. **enc–CNN**: the ProstT5 **encoder only** + a tiny **CNN head** (`cnn_chkpnt_AA_CNN/model.pt`) that emits per-residue AA logits in **one parallel pass**.

For a length-stratified test set drawn from AlphaFoldDB, we measure:

- **Latency** (s/protein, end-to-end median over repeats)
- **Throughput** (tokens/s = generated AA tokens / wall time)
- **Peak vRAM** (GB)
- **enc–dec ↔ enc–CNN agreement** (per-residue AA identity)

Pipeline outline:

1. Determine the AlphaFoldDB test set (varying lengths)
2. Get the ProstT5 encoder–decoder (and reuse its encoder for the enc–CNN path)
3. Use the CNN checkpoint (`model.pt`) for the enc–CNN path
4. Measure latency, throughput, peak vRAM for both pipelines, plus agreement

In [ ]:
%pip install tiktoken sentencepiece torch pyhmmer
%pip install 'accelerate>=0.26.0'
%pip install "transformers==4.46.3" "protobuf>=5.27" sentencepiece

In [ ]:
#@title Imports + device check. { display-mode: "form" }
import os, time, json, gc, statistics, subprocess, shutil
from pathlib import Path

import torch
import torch.nn as nn
from transformers import T5Tokenizer, AutoModelForSeq2SeqLM

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"torch={torch.__version__}  device={device}  "
      f"cuda={torch.cuda.is_available()}")
if device.type == "cpu":
    print("WARNING: running on CPU — timings will not reflect realistic GPU latency.")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive')

In [ ]:
#@title Configuration. { display-mode: "form" }
PROSTT5_NAME = "Rostlab/ProstT5_fp16"

NOTEBOOK_DIR = Path(".").resolve()

CNN_CKPT = NOTEBOOK_DIR / "model.pt"
CNN_CKPT = Path("/content/drive/MyDrive/model.pt")
# CNN_CKPT = NOTEBOOK_DIR / "cnn_chkpnt_AA_CNN" / "model.pt"
DATA_DIR = NOTEBOOK_DIR / "benchmark_data"
RESULTS_DIR = NOTEBOOK_DIR / "benchmark_results"
DATA_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# Benchmark protocol
NUM_WARMUP = 2          # untimed model.generate calls before the timed ones
NUM_REPEATS = 3         # timed repeats per protein (we report median)
USE_FP16 = True         # ProstT5 ships an fp16 checkpoint; CNN is small so we cast its inputs to match

ALLOW_CPU_FALLBACK = True



# AA vocabulary order used by the ProstT5 AA-CNN head (20 std amino acids, alphabetical).
AA_LETTERS = "ACDEFGHIKLMNPQRSTVWY"
assert len(AA_LETTERS) == 20

print(f"CNN checkpoint exists: {CNN_CKPT.exists()}  -> {CNN_CKPT}")

In [ ]:
!nvidia-smi

In [ ]:
#@title Load ProstT5 (tokenizer + full encoder–decoder). { display-mode: "form" }
tokenizer = T5Tokenizer.from_pretrained(PROSTT5_NAME, do_lower_case=False, legacy=True)

dtype = torch.float16 if (USE_FP16 and device.type == "cuda") else torch.float32
model = AutoModelForSeq2SeqLM.from_pretrained(
    PROSTT5_NAME,
    low_cpu_mem_usage=True,
    torch_dtype=dtype,
).to(device).eval()
if dtype == torch.float16:
    model = model.half()

# We will reuse the same encoder for the enc-CNN path (no second copy of the weights).
encoder = model.get_encoder()

print(f"ProstT5 loaded. dtype={next(model.parameters()).dtype}  "
      f"params(M)={sum(p.numel() for p in model.parameters())/1e6:.1f}")

In [ ]:
#@title Define + load the AA CNN head. { display-mode: "form" }
class AACNN(nn.Module):
    """ProstT5 CNN head for 3Di -> AA prediction.

    Matches the checkpoint shapes:
      classifier.0: Conv2d(1024 -> 32, kernel=(7,1), padding=(3,0))
      classifier.3: Conv2d(  32 -> 20, kernel=(7,1), padding=(3,0))
    Input  : per-residue encoder embeddings, shape (B, L, 1024)
    Output : per-residue AA logits,         shape (B, L, 20)
    """

    def __init__(self, num_tokens: int = 20, hidden: int = 32, in_dim: int = 1024):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Conv2d(in_dim, hidden, kernel_size=(7, 1), padding=(3, 0)),
            nn.ReLU(),
            nn.Dropout(0.0),
            nn.Conv2d(hidden, num_tokens, kernel_size=(7, 1), padding=(3, 0)),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (B, L, C) -> (B, C, L, 1)
        x = x.permute(0, 2, 1).unsqueeze(-1)
        x = self.classifier(x)
        # (B, V, L, 1) -> (B, L, V)
        return x.squeeze(-1).permute(0, 2, 1)


cnn = AACNN(num_tokens=20).to(device).eval()
ckpt = torch.load(CNN_CKPT, map_location=device, weights_only=False)
state_dict = ckpt.get("state_dict", ckpt)
missing, unexpected = cnn.load_state_dict(state_dict, strict=True)
print(f"CNN loaded. missing={missing} unexpected={unexpected}")
print(f"CNN params: {sum(p.numel() for p in cnn.parameters()):,}")

## Step 1 — Length-stratified test set from AlphaFoldDB

Pick a small set of UniProt IDs covering a range of protein lengths (e.g., ~50, 100, 200, 400, 800 residues). For each ID we:

1. Download the AlphaFold predicted structure (`AF-<ID>-F1-model_v4.pdb`).
2. Run **Foldseek** to extract the 1D **3Di** structural sequence.
3. Cache the 3Di in a single FASTA file so all benchmark runs use the same inputs.

If you cannot run `foldseek` here (e.g., on macOS without the right binary), drop a pre-built FASTA at `benchmark_data/test_set_3Di.fasta` with `>ID` headers and lower-case 3Di sequences and the next cell will pick it up.

In [ ]:
#@title Length-bucketed UniProt IDs. { display-mode: "form" }
# 5 length buckets x 1-2 proteins each. Edit / extend as you like;
# the rest of the notebook just iterates over whatever is in TEST_IDS.
# (Approximate lengths shown for orientation; real lengths come from Foldseek output.)
TEST_IDS = [
    # ~50–80 residues
    "P0DTC9",   # SARS-CoV-2 nucleocapsid fragment-like; short
    # ~100–150 residues
    "P01308",   # human insulin (preproinsulin, ~110)
    # ~200–300 residues
    "P00533",   # EGFR (truncated AFDB entry varies; placeholder)
    # ~400–500 residues
    "A0A6G0XC32",  # the example used in the original ProstT5 notebook
    # ~700–900 residues
    "P04637",   # human p53 (~393); replace with longer if you want
]
print(f"#proteins requested: {len(TEST_IDS)}")
print(TEST_IDS)

In [ ]:
#@title Build the test-set 3Di FASTA (download AFDB + run Foldseek). { display-mode: "form" }
import platform, urllib.request, tarfile, stat

TEST_FASTA = DATA_DIR / "test_set_3Di.fasta"
TEST_AA_FASTA = DATA_DIR / "test_set_AA.fasta"
FOLDSEEK_DIR = NOTEBOOK_DIR / "foldseek_bin"


def _foldseek_url() -> str | None:
    """Return the right Foldseek release tarball for this platform, or None if unknown."""
    sysname = platform.system()
    arch = platform.machine().lower()
    if sysname == "Linux":
        return "https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz"
    if sysname == "Darwin":
        return "https://mmseqs.com/foldseek/foldseek-osx-universal.tar.gz"
    return None


def _have_foldseek() -> str | None:
    """Return path to a usable foldseek binary, or None.
    Searches PATH, the notebook dir, /content/ (Colab), /tmp/, and any 'foldseek/bin/foldseek'
    under the cwd or notebook dir. Order: explicit known locations first, then glob.
    """
    on_path = shutil.which("foldseek")
    if on_path:
        return on_path
    candidates = [
        FOLDSEEK_DIR / "bin" / "foldseek",
        NOTEBOOK_DIR / "foldseek" / "bin" / "foldseek",
        Path("/content/foldseek/bin/foldseek"),
        Path.cwd() / "foldseek" / "bin" / "foldseek",
        Path("/tmp/foldseek/bin/foldseek"),
        Path("/usr/local/bin/foldseek"),
    ]
    for c in candidates:
        if c.exists() and os.access(c, os.X_OK):
            return str(c)
    # Last-resort glob.
    for root in {NOTEBOOK_DIR, Path.cwd(), Path("/content"), Path("/tmp")}:
        if not root.exists():
            continue
        for hit in root.glob("**/foldseek/bin/foldseek"):
            if hit.is_file() and os.access(hit, os.X_OK):
                return str(hit)
    return None


def _install_foldseek() -> str:
    """Download + extract a Foldseek release into FOLDSEEK_DIR; return path to the binary."""
    FOLDSEEK_DIR.mkdir(parents=True, exist_ok=True)
    url = _foldseek_url()
    if url is None:
        raise RuntimeError(
            f"Auto-install not supported on {platform.system()}/{platform.machine()}. "
            "Install foldseek manually and put it on PATH, "
            f"or drop pre-built FASTAs at {TEST_FASTA} / {TEST_AA_FASTA}."
        )
    tarball = FOLDSEEK_DIR / "foldseek.tar.gz"
    if not tarball.exists():
        print(f"Downloading foldseek from {url} ...")
        urllib.request.urlretrieve(url, tarball)
    print(f"Extracting {tarball} ...")
    with tarfile.open(tarball, "r:gz") as tf:
        tf.extractall(FOLDSEEK_DIR)
    # Releases extract to <FOLDSEEK_DIR>/foldseek/bin/foldseek; flatten to <FOLDSEEK_DIR>/bin/.
    src_bin = FOLDSEEK_DIR / "foldseek" / "bin" / "foldseek"
    dst_bin = FOLDSEEK_DIR / "bin" / "foldseek"
    if src_bin.exists():
        dst_bin.parent.mkdir(parents=True, exist_ok=True)
        if dst_bin.exists():
            dst_bin.unlink()
        shutil.copy(src_bin, dst_bin)
        st = dst_bin.stat()
        dst_bin.chmod(st.st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
        return str(dst_bin)
    raise RuntimeError(f"foldseek binary not found after extracting {tarball}")


def _ensure_foldseek() -> str:
    fs = _have_foldseek()
    if fs is not None:
        return fs
    print("foldseek not found in any known location; auto-installing ...")
    fs = _install_foldseek()
    print(f"foldseek installed at: {fs}")
    return fs


def _afdb_pdb_url(uniprot_id: str) -> str:
    """Resolve the canonical AlphaFoldDB PDB URL for a UniProt ID via the AFDB API.
    Robust to AFDB version bumps (they go from model_v4 -> model_v6 etc.).
    """
    import json
    api = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"
    with urllib.request.urlopen(api, timeout=15) as r:
        body = json.loads(r.read())
    if not body or not isinstance(body, list):
        raise RuntimeError(f"AFDB API returned no entries for {uniprot_id}")
    pdb_url = body[0].get("pdbUrl")
    if not pdb_url:
        raise RuntimeError(f"AFDB API entry for {uniprot_id} has no pdbUrl")
    return pdb_url


def _download_pdb(uniprot_id: str, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    pdb = out_dir / f"AF-{uniprot_id}-F1-model.pdb"
    if pdb.exists() and pdb.stat().st_size > 0:
        return pdb
    url = _afdb_pdb_url(uniprot_id)
    urllib.request.urlretrieve(url, pdb)
    return pdb


def _extract_3di(uniprot_id: str, pdb: Path, foldseek: str) -> tuple[str, str]:
    """Returns (aa_seq, three_di_seq_lowercase)."""
    work = DATA_DIR / "foldseek_work" / uniprot_id
    work.mkdir(parents=True, exist_ok=True)
    pdb_dir = work / "pdbs"
    pdb_dir.mkdir(exist_ok=True)
    target = pdb_dir / pdb.name
    if not target.exists():
        shutil.copy(pdb, target)
    db = work / "queryDB"
    subprocess.run([foldseek, "createdb", str(pdb_dir), str(db)], check=True)
    subprocess.run([foldseek, "lndb", str(db) + "_h", str(db) + "_ss_h"], check=True)
    aa_fasta = work / "queryDB.fasta"
    di_fasta = work / "queryDB_ss.fasta"
    subprocess.run([foldseek, "convert2fasta", str(db),       str(aa_fasta)], check=True)
    subprocess.run([foldseek, "convert2fasta", str(db) + "_ss", str(di_fasta)], check=True)

    def read_one(p: Path, lower: bool) -> str:
        seq = []
        for line in p.read_text().splitlines():
            if line.startswith(">"): continue
            seq.append(line.strip())
        s = "".join(seq).replace("-", "")
        return s.lower() if lower else s

    return read_one(aa_fasta, lower=False), read_one(di_fasta, lower=True)


def build_test_set(ids: list[str]) -> dict[str, dict]:
    """Returns {uid: {'aa': ..., '3di': ..., 'length': ...}}, also writes FASTAs to disk."""
    if TEST_FASTA.exists() and TEST_AA_FASTA.exists():
        print(f"Reusing cached FASTA: {TEST_FASTA}")
        out: dict[str, dict] = {}
        # AA
        cur = None
        for line in TEST_AA_FASTA.read_text().splitlines():
            if line.startswith(">"):
                cur = line[1:].split()[0]
                out.setdefault(cur, {})["aa"] = ""
            elif cur:
                out[cur]["aa"] += line.strip()
        # 3Di
        cur = None
        for line in TEST_FASTA.read_text().splitlines():
            if line.startswith(">"):
                cur = line[1:].split()[0]
                out.setdefault(cur, {})["3di"] = ""
            elif cur:
                out[cur]["3di"] += line.strip().lower()
        for uid, rec in out.items():
            rec["length"] = len(rec.get("3di", ""))
        return out

    foldseek = _ensure_foldseek()

    out: dict[str, dict] = {}
    pdb_root = DATA_DIR / "afdb_pdbs"
    for uid in ids:
        try:
            pdb = _download_pdb(uid, pdb_root)
            aa, di = _extract_3di(uid, pdb, foldseek)
            if not aa or not di or len(aa) != len(di):
                print(f"  SKIP {uid}: bad AA/3Di (lengths {len(aa)} vs {len(di)})")
                continue
            out[uid] = {"aa": aa, "3di": di, "length": len(di)}
            print(f"  OK   {uid}  L={len(di):4d}")
        except Exception as e:
            print(f"  FAIL {uid}: {e}")

    with TEST_AA_FASTA.open("w") as f:
        for uid, rec in out.items():
            f.write(f">{uid}\n{rec['aa']}\n")
    with TEST_FASTA.open("w") as f:
        for uid, rec in out.items():
            f.write(f">{uid}\n{rec['3di']}\n")
    return out


test_set = build_test_set(TEST_IDS)
print(f"\nFinal test set: {len(test_set)} proteins")
for uid, rec in sorted(test_set.items(), key=lambda kv: kv[1]["length"]):
    print(f"  {uid:>15s}  L={rec['length']:4d}")

## Step 2 — Helpers

We define one helper per pipeline. Both helpers:

- Use `@torch.inference_mode()` (no autograd, lower memory).
- Run `NUM_WARMUP` untimed calls before the timed ones (CUDA kernel autotune, KV-cache allocation, etc., easily inflate the first call by 10×).
- Use `torch.cuda.synchronize()` around every timed region so we measure the GPU, not the CUDA queue.
- Reset and read `torch.cuda.max_memory_allocated()` per protein for a clean per-protein vRAM number.
- Return one row per (protein, repeat).

Inverse-folding prompt convention for ProstT5: prefix `<fold2AA>`, lower-case 3Di letters, space-separated tokens.

In [ ]:
#@title Helper: timing utilities. { display-mode: "form" }
def _sync():
    if device.type == "cuda":
        torch.cuda.synchronize()


def _reset_peak_mem():
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)


def _peak_mem_gb() -> float:
    if device.type != "cuda":
        return 0.0
    return torch.cuda.max_memory_allocated(device) / 1e9


def _format_3di(seq: str) -> str:
    """ProstT5 3Di prompt: <fold2AA> + space-separated lowercase 3Di letters."""
    return "<fold2AA> " + " ".join(list(seq.lower()))


def _decode_aa(token_ids: torch.Tensor) -> str:
    """Decode a 1D tensor of T5 token ids to an AA string (drop spaces / specials)."""
    s = tokenizer.decode(token_ids, skip_special_tokens=True)
    return "".join(s.split())

In [ ]:
#@title Helper: enc–dec inverse-folding (greedy, autoregressive). { display-mode: "form" }
@torch.inference_mode()
def time_encdec(uid: str, three_di: str,
                num_warmup: int = NUM_WARMUP,
                num_repeats: int = NUM_REPEATS) -> tuple[list[dict], str]:
    L = len(three_di)
    enc = tokenizer(
        [_format_3di(three_di)],
        add_special_tokens=True,
        return_tensors="pt",
    ).to(device)

    gen_kwargs = dict(
        input_ids=enc.input_ids,
        attention_mask=enc.attention_mask,
        max_length=L + 2,        # output length = L + EOS, +1 buffer
        min_length=L + 1,
        num_beams=1,
        do_sample=False,         # greedy (deterministic baseline)
        num_return_sequences=1,
    )

    # Warmup.
    for _ in range(num_warmup):
        _ = model.generate(**gen_kwargs)
    _sync()

    rows = []
    last_pred = ""
    for r in range(num_repeats):
        _reset_peak_mem()
        _sync()
        t0 = time.perf_counter()
        out = model.generate(**gen_kwargs)
        _sync()
        dt = time.perf_counter() - t0

        n_new = int(out.shape[1] - 1)  # subtract decoder start token
        pred_aa = _decode_aa(out[0])
        last_pred = pred_aa
        rows.append({
            "protein_id": uid,
            "pipeline": "enc_dec",
            "length": L,
            "repeat": r,
            "wall_s": dt,
            "tokens_per_s": n_new / dt if dt > 0 else float("nan"),
            "peak_vram_gb": _peak_mem_gb(),
            "pred_len": len(pred_aa),
        })
    return rows, last_pred

In [ ]:
#@title Helper: enc–CNN inverse-folding (encoder + CNN, one parallel pass). { display-mode: "form" }
@torch.inference_mode()
def time_enccnn(uid: str, three_di: str,
                num_warmup: int = NUM_WARMUP,
                num_repeats: int = NUM_REPEATS) -> tuple[list[dict], str]:
    L = len(three_di)
    enc = tokenizer(
        [_format_3di(three_di)],
        add_special_tokens=True,
        return_tensors="pt",
    ).to(device)

    def _forward() -> torch.Tensor:
        h = encoder(
            input_ids=enc.input_ids,
            attention_mask=enc.attention_mask,
        ).last_hidden_state                  # (1, T, 1024) — T includes <fold2AA> + EOS
        # Drop the prefix token (1) and trailing EOS (1) to align to L residues.
        h = h[:, 1:-1, :]
        # CNN runs in fp32 for numerical safety; cost is negligible vs. the encoder.
        logits = cnn(h.float())              # (1, L, 20)
        return logits

    # Warmup.
    for _ in range(num_warmup):
        _ = _forward()
    _sync()

    rows = []
    last_pred = ""
    for r in range(num_repeats):
        _reset_peak_mem()
        _sync()
        t0 = time.perf_counter()
        logits = _forward()
        ids = logits.argmax(-1)              # (1, L)
        _sync()
        dt = time.perf_counter() - t0

        pred_aa = "".join(AA_LETTERS[int(i)] for i in ids[0].tolist())
        last_pred = pred_aa
        rows.append({
            "protein_id": uid,
            "pipeline": "enc_cnn",
            "length": L,
            "repeat": r,
            "wall_s": dt,
            "tokens_per_s": len(pred_aa) / dt if dt > 0 else float("nan"),
            "peak_vram_gb": _peak_mem_gb(),
            "pred_len": len(pred_aa),
        })
    return rows, last_pred

## Step 3 — Run benchmarks

Iterate over the test set, run both pipelines, store one row per (protein, pipeline, repeat) and one prediction per (protein, pipeline) for the agreement check.

In [ ]:
#@title Run both pipelines on the full test set. { display-mode: "form" }
all_rows: list[dict] = []
predictions: dict[str, dict[str, str]] = {}  # {uid: {"enc_dec": ..., "enc_cnn": ...}}

for uid, rec in sorted(test_set.items(), key=lambda kv: kv[1]["length"]):
    L = rec["length"]
    print(f"--- {uid}  L={L} ---")

    rows_dec, pred_dec = time_encdec(uid, rec["3di"])
    print(f"  enc_dec: median {statistics.median(r['wall_s'] for r in rows_dec):.3f}s "
          f"peak vRAM {max(r['peak_vram_gb'] for r in rows_dec):.2f} GB")
    rows_cnn, pred_cnn = time_enccnn(uid, rec["3di"])
    print(f"  enc_cnn: median {statistics.median(r['wall_s'] for r in rows_cnn):.3f}s "
          f"peak vRAM {max(r['peak_vram_gb'] for r in rows_cnn):.2f} GB")

    all_rows.extend(rows_dec)
    all_rows.extend(rows_cnn)
    predictions[uid] = {"enc_dec": pred_dec, "enc_cnn": pred_cnn, "ground_truth_aa": rec["aa"]}

    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

print(f"\nTotal timed runs: {len(all_rows)}")

In [ ]:
#@title Aggregate + persist results. { display-mode: "form" }
import pandas as pd

raw = pd.DataFrame(all_rows)
raw.to_csv(RESULTS_DIR / "raw_runs.csv", index=False)

summary = (
    raw.groupby(["protein_id", "pipeline", "length"])
       .agg(latency_s_median=("wall_s", "median"),
            latency_s_min=("wall_s", "min"),
            tokens_per_s=("tokens_per_s", "median"),
            peak_vram_gb=("peak_vram_gb", "max"))
       .reset_index()
       .sort_values(["pipeline", "length"])
)
summary.to_csv(RESULTS_DIR / "summary_per_protein.csv", index=False)

print("=== Summary (median over repeats) ===")
with pd.option_context("display.float_format", "{:.4f}".format):
    print(summary.to_string(index=False))

In [ ]:
#@title Speedup vs. enc–dec, per protein. { display-mode: "form" }
pivot = summary.pivot(index=["protein_id", "length"],
                      columns="pipeline",
                      values="latency_s_median").reset_index()
pivot["speedup_enc_cnn_over_enc_dec"] = pivot["enc_dec"] / pivot["enc_cnn"]
pivot = pivot.sort_values("length")
pivot.to_csv(RESULTS_DIR / "speedup.csv", index=False)
with pd.option_context("display.float_format", "{:.3f}".format):
    print(pivot.to_string(index=False))

## Step 4 — Agreement: enc–dec ↔ enc–CNN

Per-residue AA identity between the two pipelines' predictions, on the *same* 3Di input. This is the "how well do enc–dec and enc–CNN agree?" number the project description explicitly asks for, and it is also a useful upper bound on the acceptance rate α we will get when we use the enc–CNN as the **drafter** in the speculative-decoding step of the project.

In [ ]:
#@title Compute per-residue agreement. { display-mode: "form" }
agree_rows = []
for uid, p in predictions.items():
    a, b, gt = p["enc_dec"], p["enc_cnn"], p["ground_truth_aa"]
    n = min(len(a), len(b))
    if n == 0:
        continue
    agree = sum(1 for i in range(n) if a[i] == b[i]) / n
    # Sequence recovery vs. AlphaFold AA (a sanity check, not a project requirement).
    n_gt = min(n, len(gt))
    rec_dec = sum(1 for i in range(n_gt) if a[i] == gt[i]) / n_gt if n_gt else float("nan")
    rec_cnn = sum(1 for i in range(n_gt) if b[i] == gt[i]) / n_gt if n_gt else float("nan")
    agree_rows.append({
        "protein_id": uid,
        "length": len(a),
        "encdec_vs_enccnn_identity": agree,
        "encdec_seq_recovery_vs_AFDB": rec_dec,
        "enccnn_seq_recovery_vs_AFDB": rec_cnn,
    })

agree_df = pd.DataFrame(agree_rows).sort_values("length")
agree_df.to_csv(RESULTS_DIR / "agreement.csv", index=False)
with pd.option_context("display.float_format", "{:.3f}".format):
    print(agree_df.to_string(index=False))
print(f"\nMean enc-dec ↔ enc-CNN agreement: "
      f"{agree_df['encdec_vs_enccnn_identity'].mean():.3f}")

In [ ]:
#@title Plots: latency / throughput / vRAM vs. length. { display-mode: "form" }
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, ylabel, log in [
    (axes[0], "latency_s_median", "Latency (s/protein)", True),
    (axes[1], "tokens_per_s",     "Throughput (tokens/s)", True),
    (axes[2], "peak_vram_gb",     "Peak vRAM (GB)", False),
]:
    for pipeline, sub in summary.groupby("pipeline"):
        sub = sub.sort_values("length")
        ax.plot(sub["length"], sub[metric], "o-", label=pipeline)
    ax.set_xlabel("Protein length (residues)")
    ax.set_ylabel(ylabel)
    if log: ax.set_yscale("log")
    ax.set_xscale("log")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()

fig.suptitle("Inverse-folding baseline (3Di → AA): enc–dec vs. enc–CNN")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "baseline_plots.png", dpi=150)
plt.show()

## Step 5 — Profile-HMM drafter

This section benchmarks two HMM drafters for inverse folding on the same ProstT5 verifier.

- `HMMDrafter`: naive, prefix-blind, one-shot alignment.
- `PrefixAwareHMMDrafter`: re-anchors the HMM after each verified block.

The default case list is mixed but conservative to reduce the chance of running back into the earlier GPU OOM failure. Once the section is stable on your runtime, you can raise `HMM_MAX_CASE_LENGTH` or extend `HMM_CASE_SPECS`.

In [ ]:
#@title Install HMM dependencies. { display-mode: "form" }
%pip install -q -r ./requirements-hmm.txt

In [ ]:
#@title Build / refresh PF00535 assets and assemble HMM cases. { display-mode: "form" }
import sys
import subprocess

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from hmm_drafter import (
    HMMDrafter,
    PrefixAwareHMMDrafter,
    encdec_greedy_reference,
    spec_decode_greedy,
)

HMM_HMM_PATH = NOTEBOOK_DIR / "hmm_data" / "PF00535.hmm"
if not HMM_HMM_PATH.exists():
    subprocess.run(
        [sys.executable, str(NOTEBOOK_DIR / "build_hmm.py")],
        cwd=NOTEBOOK_DIR,
        check=True,
    )
print(f"HMM artifact: {HMM_HMM_PATH}  exists={HMM_HMM_PATH.exists()}")

HMM_K_VALUES = [1, 2, 4, 8, 16]
HMM_REPEATS = 1
HMM_WARMUP = 0
HMM_MAX_CASE_LENGTH = 450
HMM_CASE_SPECS = [
    {"uid": "P39621", "label": "PF00535 in-family SpsA"},
    {"uid": "A0A6G0XC32", "label": "baseline medium target"},
    {"uid": "P04637", "label": "out-of-family p53"},
    {"uid": "P01308", "label": "short control"},
]


def _get_hmm_case(uid: str) -> dict:
    if uid in test_set and all(key in test_set[uid] for key in ("aa", "3di")):
        rec = dict(test_set[uid])
    else:
        foldseek = _ensure_foldseek()
        pdb = _download_pdb(uid, DATA_DIR / "afdb_pdbs")
        aa, di = _extract_3di(uid, pdb, foldseek)
        rec = {"aa": aa, "3di": di, "length": len(di)}
    if len(rec["aa"]) != len(rec["3di"]):
        raise ValueError(
            f"{uid}: AA and 3Di lengths disagree ({len(rec['aa'])} vs {len(rec['3di'])})"
        )
    rec["uid"] = uid
    return rec


hmm_cases = []
for spec in HMM_CASE_SPECS:
    rec = _get_hmm_case(spec["uid"])
    if rec["length"] > HMM_MAX_CASE_LENGTH:
        print(
            f"SKIP {spec['uid']}: length {rec['length']} > "
            f"HMM_MAX_CASE_LENGTH={HMM_MAX_CASE_LENGTH}"
        )
        continue
    rec["label"] = spec["label"]
    hmm_cases.append(rec)

print(f"Selected {len(hmm_cases)} HMM cases:")
for rec in hmm_cases:
    print(f"  {rec['uid']:>12s}  L={rec['length']:4d}  {rec['label']}")

In [ ]:
#@title Run HMM drafter benchmarks. { display-mode: "form" }
@torch.inference_mode()
def time_hmm_spec(
    uid: str,
    rec: dict,
    drafter_cls,
    k: int,
    ref_ids: list[int],
    num_warmup: int = HMM_WARMUP,
    num_repeats: int = HMM_REPEATS,
) -> tuple[list[dict], str]:
    variant = "prefix_aware" if drafter_cls is PrefixAwareHMMDrafter else "naive"
    rows = []
    last_pred = ""

    for _ in range(num_warmup):
        warm_drafter = drafter_cls(HMM_HMM_PATH, rec["aa"], tokenizer)
        _ = spec_decode_greedy(
            model,
            tokenizer,
            rec["3di"],
            warm_drafter,
            K=k,
            device=device,
        )
    _sync()

    for repeat in range(num_repeats):
        drafter = drafter_cls(HMM_HMM_PATH, rec["aa"], tokenizer)
        _reset_peak_mem()
        _sync()
        t0 = time.perf_counter()
        spec_ids, stats = spec_decode_greedy(
            model,
            tokenizer,
            rec["3di"],
            drafter,
            K=k,
            device=device,
        )
        _sync()
        dt = time.perf_counter() - t0

        pred_aa = "".join(tokenizer.decode(spec_ids, skip_special_tokens=True).split())
        last_pred = pred_aa
        accepted = int(stats["accepted"])
        proposed = int(stats["proposed"])
        rows.append({
            "protein_id": uid,
            "label": rec["label"],
            "pipeline": f"hmm_{variant}",
            "variant": variant,
            "k": k,
            "length": rec["length"],
            "repeat": repeat,
            "bit_exact": spec_ids == ref_ids,
            "accepted": accepted,
            "proposed": proposed,
            "accept_rate": (accepted / proposed) if proposed else 0.0,
            "steps": int(stats["steps"]),
            "extra_tokens": int(stats["extra_tokens"]),
            "reanchor_calls": int(getattr(drafter, "reanchor_calls", 0)),
            "wall_s": dt,
            "tokens_per_s": len(spec_ids) / dt if dt > 0 else float("nan"),
            "peak_vram_gb": _peak_mem_gb(),
            "pred_len": len(pred_aa),
        })
    return rows, last_pred


hmm_all_rows = []
hmm_predictions: dict[str, dict[str, str]] = {}

for rec in hmm_cases:
    ref_ids = encdec_greedy_reference(model, tokenizer, rec["3di"], device)
    ref_aa = "".join(tokenizer.decode(ref_ids, skip_special_tokens=True).split())
    hmm_predictions.setdefault(rec["uid"], {})["enc_dec_ref"] = ref_aa

    print(f"--- {rec['uid']}  L={rec['length']}  {rec['label']} ---")
    for drafter_cls in (HMMDrafter, PrefixAwareHMMDrafter):
        variant = "prefix_aware" if drafter_cls is PrefixAwareHMMDrafter else "naive"
        for k in HMM_K_VALUES:
            rows, pred = time_hmm_spec(rec["uid"], rec, drafter_cls, k, ref_ids)
            hmm_all_rows.extend(rows)
            hmm_predictions[rec["uid"]][f"{variant}_K{k}"] = pred
            row = rows[0]
            print(
                f"  {variant:13s} K={k:2d}  exact={row['bit_exact']}  "
                f"accept={row['accept_rate']:.3f}  wall={row['wall_s']:.3f}s  "
                f"tok/s={row['tokens_per_s']:.1f}  peak={row['peak_vram_gb']:.2f}GB  "
                f"reanchors={row['reanchor_calls']}"
            )

    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

print(f"\nTotal HMM timed runs: {len(hmm_all_rows)}")

In [ ]:
#@title Summarize HMM drafter results. { display-mode: "form" }
import pandas as pd

if not hmm_all_rows:
    raise RuntimeError("No HMM benchmark rows found. Run the previous cell first.")

hmm_raw = pd.DataFrame(hmm_all_rows)
hmm_raw.to_csv(RESULTS_DIR / "hmm_raw_runs.csv", index=False)

hmm_summary = (
    hmm_raw.groupby(["protein_id", "label", "variant", "k", "length"])
           .agg(
               bit_exact=("bit_exact", "min"),
               accept_rate=("accept_rate", "median"),
               wall_s=("wall_s", "median"),
               tokens_per_s=("tokens_per_s", "median"),
               peak_vram_gb=("peak_vram_gb", "max"),
               reanchor_calls=("reanchor_calls", "max"),
           )
           .reset_index()
           .sort_values(["protein_id", "variant", "k"])
)
hmm_summary.to_csv(RESULTS_DIR / "hmm_summary.csv", index=False)

print("=== HMM summary ===")
with pd.option_context("display.float_format", "{:.4f}".format):
    print(hmm_summary.to_string(index=False))

best_hmm = (
    hmm_summary.sort_values(["protein_id", "variant", "wall_s"])
               .groupby(["protein_id", "variant"], as_index=False)
               .first()
               .sort_values(["protein_id", "variant"])
)
best_hmm.to_csv(RESULTS_DIR / "hmm_best_by_variant.csv", index=False)

print("\n=== Fastest K per protein / variant ===")
with pd.option_context("display.float_format", "{:.4f}".format):
    print(best_hmm.to_string(index=False))

all_exact = bool(hmm_summary["bit_exact"].all())
print(f"\nAll HMM runs bit-exact vs enc-dec greedy: {all_exact}")